# Distance Feature Engineering
This notebook merges geocoded coordinates onto the full transaction dataset, calculates geospatial distance features for each HDB block, adds a mature estate flag, and applies CPI adjustment to convert all resale prices to 2024 dollars. The resulting dataset is saved as the final feature set for model retraining.

## Setup
Import required libraries and load the raw transaction dataset alongside the geocoded address lookup table produced in notebook 04.

In [3]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

df = pd.read_csv('../data/raw/hdb_resale_transactions.csv')

geocoded = pd.read_csv('../data/processed/geocoded_addresses.csv')

print(f"Transactions: {len(df):,}")
print(f"Geocoded addresses: {len(geocoded):,}")
df.head()

Transactions: 233,055
Geocoded addresses: 9,712


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


## Merge Coordinates
We merge latitude and longitude coordinates onto the full transaction dataset by joining on `block` and `street_name`. Each of the 9,712 unique blocks has a single coordinate pair which is applied to all transactions at that address.

In [4]:
df = df.merge(geocoded, on=['block', 'street_name'], how='left')

print(f"Transactions after merge: {len(df):,}")
print(f"Missing coordinates: {df['latitude'].isna().sum():,}")
print(f"Coverage: {(df['latitude'].notna().sum() / len(df) * 100):.1f}%")

Transactions after merge: 233,055
Missing coordinates: 0
Coverage: 100.0%


## Load Amenity Data
We load all amenity datasets produced in notebook 05. MRT opening dates and hawker centre completion dates are parsed to datetime immediately since they are used for temporal filtering during distance calculations.

In [17]:
mrt = pd.read_csv('../data/processed/mrt_stations_geocoded.csv')
mrt['opening_date'] = pd.to_datetime(mrt['opening_date'])

schools = pd.read_csv('../data/processed/schools_geocoded.csv')
hawkers = pd.read_csv('../data/processed/hawker_centres.csv')
hawkers['est_completion_date'] = pd.to_datetime(hawkers['est_completion_date'], errors='coerce')
malls = pd.read_csv('../data/processed/malls_geocoded.csv')
expressways = pd.read_csv('../data/processed/expressway_coords.csv')
bus_stops = pd.read_csv('../data/processed/bus_stops.csv')

print(f"MRT stations:   {len(mrt):,}")
print(f"Schools:        {len(schools):,}")
print(f"Hawker centres: {len(hawkers):,}")
print(f"Malls:          {len(malls):,}")
print(f"Expressway pts: {len(expressways):,}")
print(f"Bus stops:      {len(bus_stops):,}")

MRT stations:   143
Schools:        337
Hawker centres: 123
Malls:          116
Expressway pts: 3,319
Bus stops:      5,166


## Haversine Distance Function
The `haversine` function calculates the great-circle distance between two coordinate pairs in kilometres, accounting for the curvature of the Earth.

The `dist_to_amenities` function extends this to vectorised computation. Given one HDB block coordinate, it calculates distances to all rows in an amenity DataFrame simultaneously using numpy array operations. This avoids a nested Python loop and keeps the full calculation tractable at 233,055 transactions.

In [8]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on Earth in kilometres.
    Uses the Haversine formula which accounts for Earth's curvature.
    """
    R = 6371  # Earth's radius in kilometres

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

In [38]:
def dist_to_amenities(lat, lon, amenity_df, lat_col='latitude', lon_col='longitude'):
    """
    Calculate distances from a single point to all rows in amenity_df.
    Returns a numpy array of distances in kilometres.
    """
    lat_rad = radians(lat)
    lon_rad = radians(lon)
    
    amenity_lats = np.radians(amenity_df[lat_col].values)
    amenity_lons = np.radians(amenity_df[lon_col].values)
    
    dlat = amenity_lats - lat_rad
    dlon = amenity_lons - lon_rad
    
    a = np.sin(dlat/2)**2 + np.cos(lat_rad) * np.cos(amenity_lats) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    return 6371 * c

lat, lon = df.iloc[0]['latitude'], df.iloc[0]['longitude']
distances = dist_to_amenities(lat, lon, mrt)
print(f"Distance to nearest MRT: {distances.min():.3f} km")
print(f"MRT stations within 1km: {(distances < 1).sum()}")

Distance to nearest MRT: 1.004 km
MRT stations within 1km: 0


## Opening Date Validation
We convert transaction months to datetime and verify the MRT opening date filter on a sample transaction. In January 2017, 41 of the 143 MRT stations had not yet opened and are correctly excluded from that transaction's distance calculations.

In [37]:
df['transaction_date'] = pd.to_datetime(df['month'])

transaction_date = df.iloc[0]['transaction_date']
mrt_available = mrt[mrt['opening_date'] <= transaction_date]
print(f"Transaction date: {transaction_date.date()}")
print(f"MRT stations available at that date: {len(mrt_available)}")
print(f"Total MRT stations: {len(mrt)}")
print(f"Stations not yet open: {len(mrt) - len(mrt_available)}")

Transaction date: 2017-01-01
MRT stations available at that date: 102
Total MRT stations: 143
Stations not yet open: 41


## Distance Feature Calculation
We iterate over all 233,055 transactions and calculate 15 geospatial features per transaction:

**Distance features:** `dist_nearest_mrt`, `dist_nearest_school`, `dist_nearest_primary_school`, `dist_nearest_mall`, `dist_nearest_hawker`, `dist_to_cbd`, `dist_nearest_expressway`, `dist_nearest_bus_stop`

**Count features:** `num_mrt_within_1km`, `num_mrt_within_2km`, `num_schools_within_1km`, `num_primary_schools_within_1km`, `num_malls_within_2km`, `num_hawkers_within_500m`, `num_bus_stops_within_300m`

Opening date filtering is applied to MRT stations and hawker centres to prevent data leakage. Amenities that did not exist at the time of a transaction are excluded from that transaction's calculations. For other amenities, reliable historical opening date data was not publicly available. Since the vast majority of schools, malls, and bus stops predate our 2017 dataset period, the impact is minimal and is noted as a project limitation.

Radius thresholds are chosen based on Singapore-specific context:
- MRT 1km and 2km — captures walkable and near-walkable access
- Primary schools 1km — matches MOE's Phase 2B/2C registration priority zone
- Malls 2km — destination amenity, not walkable
- Hawker centres 500m — daily-use amenity, comfortable walking distance
- Bus stops 300m — standard walkable catchment for public transport planning

In [39]:
dist_nearest_mrt = []
num_mrt_within_1km = []
num_mrt_within_2km = []
dist_nearest_school = []
num_schools_within_1km = []
dist_nearest_primary_school = []
num_primary_schools_within_1km = []
dist_nearest_mall = []
num_malls_within_2km = []
dist_nearest_hawker = []
num_hawkers_within_500m = []
dist_to_cbd = []
dist_nearest_expressway = []
dist_nearest_bus_stop = []
num_bus_stops_within_300m = []

# CBD coordinates — Raffles Place
CBD_LAT, CBD_LON = 1.2830, 103.8513

# Primary school filter
primary_schools = schools[schools['mainlevel_code'].isin(['PRIMARY', 'MIXED LEVEL (P1-S4)'])].copy()

print(f"Primary schools: {len(primary_schools)}")
print(f"All schools: {len(schools)}")

Primary schools: 182
All schools: 337


In [23]:
for idx, row in df.iterrows():
    lat, lon = row['latitude'], row['longitude']
    transaction_date = row['transaction_date']

    # MRT — filter by opening date
    mrt_available = mrt[mrt['opening_date'] <= transaction_date]
    mrt_dists = dist_to_amenities(lat, lon, mrt_available)
    dist_nearest_mrt.append(mrt_dists.min() if len(mrt_dists) > 0 else np.nan)
    num_mrt_within_1km.append((mrt_dists < 1).sum())
    num_mrt_within_2km.append((mrt_dists < 2).sum())

    # Schools
    school_dists = dist_to_amenities(lat, lon, schools)
    dist_nearest_school.append(school_dists.min())
    num_schools_within_1km.append((school_dists < 1).sum())

    # Primary schools
    pri_dists = dist_to_amenities(lat, lon, primary_schools)
    dist_nearest_primary_school.append(pri_dists.min())
    num_primary_schools_within_1km.append((pri_dists < 1).sum())

    # Malls
    mall_dists = dist_to_amenities(lat, lon, malls)
    dist_nearest_mall.append(mall_dists.min())
    num_malls_within_2km.append((mall_dists < 2).sum())

    # Hawker centres — filter by completion date
    hawkers_available = hawkers[
        hawkers['est_completion_date'].isna() | 
        (hawkers['est_completion_date'] <= transaction_date)
    ]
    hawker_dists = dist_to_amenities(lat, lon, hawkers_available)
    dist_nearest_hawker.append(hawker_dists.min() if len(hawker_dists) > 0 else np.nan)
    num_hawkers_within_500m.append((hawker_dists < 0.5).sum())

    # CBD distance
    dist_to_cbd.append(haversine(lat, lon, CBD_LAT, CBD_LON))

    # Expressways
    exp_dists = dist_to_amenities(lat, lon, expressways)
    dist_nearest_expressway.append(exp_dists.min())

    # Bus stops
    bus_dists = dist_to_amenities(lat, lon, bus_stops)
    dist_nearest_bus_stop.append(bus_dists.min())
    num_bus_stops_within_300m.append((bus_dists < 0.3).sum())

    if (idx + 1) % 10000 == 0:
        print(f"Processed {idx + 1:,} / {len(df):,}")

print("Done.")

Processed 10,000 / 233,055
Processed 20,000 / 233,055
Processed 30,000 / 233,055
Processed 40,000 / 233,055
Processed 50,000 / 233,055
Processed 60,000 / 233,055
Processed 70,000 / 233,055
Processed 80,000 / 233,055
Processed 90,000 / 233,055
Processed 100,000 / 233,055
Processed 110,000 / 233,055
Processed 120,000 / 233,055
Processed 130,000 / 233,055
Processed 140,000 / 233,055
Processed 150,000 / 233,055
Processed 160,000 / 233,055
Processed 170,000 / 233,055
Processed 180,000 / 233,055
Processed 190,000 / 233,055
Processed 200,000 / 233,055
Processed 210,000 / 233,055
Processed 220,000 / 233,055
Processed 230,000 / 233,055
Done.


In [24]:
print(f"Length of results: {len(dist_nearest_mrt):,}")
print(f"\nSample values:")
print(f"dist_nearest_mrt:       {dist_nearest_mrt[:3]}")
print(f"dist_nearest_hawker:    {dist_nearest_hawker[:3]}")
print(f"dist_to_cbd:            {dist_to_cbd[:3]}")
print(f"num_mrt_within_1km:     {num_mrt_within_1km[:3]}")
print(f"num_hawkers_within_500m:{num_hawkers_within_500m[:3]}")

Length of results: 233,055

Sample values:
dist_nearest_mrt:       [np.float64(1.0039959848633804), np.float64(1.267605333327398), np.float64(1.0711775793770115)]
dist_nearest_hawker:    [np.float64(0.1724193099680737), np.float64(0.4105462439026658), np.float64(0.5855211609152101)]
dist_to_cbd:            [8.789584168467353, 9.889190833168252, 11.008129126934422]
num_mrt_within_1km:     [np.int64(0), np.int64(0), np.int64(0)]
num_hawkers_within_500m:[np.int64(1), np.int64(3), np.int64(0)]


## Assign Distance Features
The computed lists are assigned as new columns on the transaction DataFrame.

In [35]:
df['dist_nearest_mrt'] = dist_nearest_mrt
df['num_mrt_within_1km'] = num_mrt_within_1km
df['num_mrt_within_2km'] = num_mrt_within_2km
df['dist_nearest_school'] = dist_nearest_school
df['num_schools_within_1km'] = num_schools_within_1km
df['dist_nearest_primary_school'] = dist_nearest_primary_school
df['num_primary_schools_within_1km'] = num_primary_schools_within_1km
df['dist_nearest_mall'] = dist_nearest_mall
df['num_malls_within_2km'] = num_malls_within_2km
df['dist_nearest_hawker'] = dist_nearest_hawker
df['num_hawkers_within_500m'] = num_hawkers_within_500m
df['dist_to_cbd'] = dist_to_cbd
df['dist_nearest_expressway'] = dist_nearest_expressway
df['dist_nearest_bus_stop'] = dist_nearest_bus_stop
df['num_bus_stops_within_300m'] = num_bus_stops_within_300m

print("Features added.")
print(f"DataFrame shape: {df.shape}")


Features added.
DataFrame shape: (233055, 32)


## Mature Estate Flag
We add a binary `is_mature_estate` column based on HDB's official classification of mature and non-mature towns. Mature estates generally have better infrastructure, more established amenities, and command higher resale prices.

In [26]:
mature_estates = [
    'ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT MERAH', 'BUKIT TIMAH',
    'CENTRAL AREA', 'CLEMENTI', 'GEYLANG', 'KALLANG/WHAMPOA',
    'MARINE PARADE', 'PASIR RIS', 'QUEENSTOWN', 'SERANGOON',
    'TAMPINES', 'TOA PAYOH'
]

df['is_mature_estate'] = df['town'].isin(mature_estates).astype(int)

print(f"Mature estate transactions: {df['is_mature_estate'].sum():,}")
print(f"Non-mature estate transactions: {(df['is_mature_estate'] == 0).sum():,}")

Mature estate transactions: 97,253
Non-mature estate transactions: 135,802


## CPI Adjustment
Resale prices across our dataset span 2017 to 2026 and are expressed in nominal dollars of their respective years. A flat sold for SGD 400,000 in 2017 is not directly comparable to one sold for SGD 400,000 in 2024 due to inflation.

We source monthly CPI data (All Items, base year 2024) from the Singapore Department of Statistics via data.gov.sg. The adjustment formula is:

`resale_price_real = resale_price × (BASE_CPI / transaction_CPI)`

where `BASE_CPI` is the 2024 annual average. This converts all prices to 2024 dollars, making historical transactions internally consistent for model training.

CPI data was unavailable for May and June 2026 at the time of writing. These 3,006 transactions are filled with the most recent available value (April 2026 = 102.052), introducing negligible error given the low month-to-month CPI volatility.

In [28]:
cpi = pd.read_csv('../data/raw/cpi.csv')
print(cpi.shape)
print(cpi.head(10))
print(cpi.dtypes)

(207, 785)
                                          DataSeries  2026Apr  2026Mar  \
0                                          All Items  102.052  102.405   
1                                               Food  102.730  102.521   
2         Food Excl Food & Beverage Serving Services  103.626  103.319   
3                             Rice & Cereal Products  103.774  102.620   
4                                               Rice  103.602   99.800   
5                                              Flour  102.328   96.770   
6                                              Bread  103.339  102.987   
7              Macaroni, Noodles & Similar Pasta ...   99.546   99.500   
8                                 Biscuits & Cookies  105.544  105.265   
9                                   Cakes & Pastries  105.311  103.983   

   2026Feb  2026Jan  2025Dec  2025Nov  2025Oct  2025Sep  2025Aug  ...  \
0  101.918  101.333  101.854  101.579  101.330  101.320  100.963  ...   
1  102.531  102.042  101.790

In [29]:
cpi_all = cpi[cpi['DataSeries'] == 'All Items'].iloc[0]

cpi_long = cpi_all.drop('DataSeries').reset_index()
cpi_long.columns = ['month_str', 'cpi']
cpi_long = cpi_long[cpi_long['cpi'] != 'na'].copy()
cpi_long['cpi'] = cpi_long['cpi'].astype(float)

print(cpi_long.head(10))
print(f"\nShape: {cpi_long.shape}")

  month_str      cpi
0   2026Apr  102.052
1   2026Mar  102.405
2   2026Feb  101.918
3   2026Jan  101.333
4   2025Dec  101.854
5   2025Nov  101.579
6   2025Oct  101.330
7   2025Sep  101.320
8   2025Aug  100.963
9   2025Jul  100.447

Shape: (784, 2)


In [30]:
cpi_long['transaction_date'] = pd.to_datetime(cpi_long['month_str'], format='%Y%b')

cpi_long = cpi_long[cpi_long['transaction_date'] >= '2017-01-01'].copy()
cpi_long = cpi_long.sort_values('transaction_date').reset_index(drop=True)

print(f"CPI records in dataset period: {len(cpi_long)}")
print(cpi_long.head())
print(cpi_long.tail())

CPI records in dataset period: 112
  month_str     cpi transaction_date
0   2017Jan  85.102       2017-01-01
1   2017Feb  85.100       2017-02-01
2   2017Mar  85.136       2017-03-01
3   2017Apr  84.877       2017-04-01
4   2017May  85.164       2017-05-01
    month_str      cpi transaction_date
107   2025Dec  101.854       2025-12-01
108   2026Jan  101.333       2026-01-01
109   2026Feb  101.918       2026-02-01
110   2026Mar  102.405       2026-03-01
111   2026Apr  102.052       2026-04-01


In [31]:
BASE_CPI = cpi_long[cpi_long['transaction_date'] >= '2024-01-01']['cpi'].mean()
print(f"Base CPI (2024 average): {BASE_CPI:.3f}")

df = df.merge(cpi_long[['transaction_date', 'cpi']], on='transaction_date', how='left')
print(f"Missing CPI values: {df['cpi'].isna().sum()}")


df['resale_price_real'] = df['resale_price'] * (BASE_CPI / df['cpi'])

print(f"\nSample comparison:")
print(df[['month', 'resale_price', 'cpi', 'resale_price_real']].head(5))

Base CPI (2024 average): 100.662
Missing CPI values: 3006

Sample comparison:
     month  resale_price     cpi  resale_price_real
0  2017-01      232000.0  85.102      274419.912911
1  2017-01      250000.0  85.102      295711.113051
2  2017-01      262000.0  85.102      309905.246477
3  2017-01      265000.0  85.102      313453.779834
4  2017-01      265000.0  85.102      313453.779834


In [32]:
missing_cpi = df[df['cpi'].isna()]['month'].value_counts().sort_index()
print(missing_cpi)

month
2026-05    2134
2026-06     872
Name: count, dtype: int64


In [33]:
latest_cpi = cpi_long['cpi'].iloc[-1]
print(f"Filling with latest CPI: {latest_cpi}")

df['cpi'] = df['cpi'].fillna(latest_cpi)
df['resale_price_real'] = df['resale_price'] * (BASE_CPI / df['cpi'])

print(f"Missing CPI values after fill: {df['cpi'].isna().sum()}")

Filling with latest CPI: 102.052
Missing CPI values after fill: 0


## Save Final Feature Set
We save the complete enriched dataset to `data/processed/features_with_geo.csv`. This file contains all original transaction columns plus 20 new features — 15 geospatial distance and count features, the mature estate flag, latitude and longitude, CPI, and the CPI-adjusted resale price.

In [34]:
df.to_csv('../data/processed/features_with_geo.csv', index=False)
print(f"Saved to data/processed/features_with_geo.csv")
print(f"Shape: {df.shape}")
print(f"\nNew columns added in this notebook:")
geo_cols = ['latitude', 'longitude', 'dist_nearest_mrt', 'num_mrt_within_1km', 
            'num_mrt_within_2km', 'dist_nearest_school', 'num_schools_within_1km',
            'dist_nearest_primary_school', 'num_primary_schools_within_1km',
            'dist_nearest_mall', 'num_malls_within_2km', 'dist_nearest_hawker',
            'num_hawkers_within_500m', 'dist_to_cbd', 'dist_nearest_expressway',
            'dist_nearest_bus_stop', 'num_bus_stops_within_300m', 
            'is_mature_estate', 'cpi', 'resale_price_real']
print(df[geo_cols].describe())

Saved to data/processed/features_with_geo.csv
Shape: (233055, 32)

New columns added in this notebook:
            latitude      longitude  dist_nearest_mrt  num_mrt_within_1km  \
count  233055.000000  233055.000000     233055.000000       233055.000000   
mean        1.368165     103.841173          0.792527            1.081358   
std         0.042994       0.070993          0.443371            0.952970   
min         1.270380     103.685228          0.036138            0.000000   
25%         1.337357     103.780381          0.466696            0.000000   
50%         1.367421     103.846208          0.707175            1.000000   
75%         1.396742     103.898666          1.028425            2.000000   
max         1.457071     103.987805          3.516004           10.000000   

       num_mrt_within_2km  dist_nearest_school  num_schools_within_1km  \
count       233055.000000        233055.000000           233055.000000   
mean             3.482822             0.334886         